# NB_patient_360

Builds the **gold.patient_360** table — one row per patient with hospital, insurance, and lab-test summary attributes. Full rebuild every run (`CREATE OR REPLACE TABLE … AS SELECT`).

In [0]:
# -- Configuration ----------------------------------------------------------
catalog = "adb_databeli"
target_table = f"{catalog}.gold.patient_360"

print(f"✅ Configuration")
print(f"   📌 Catalog      : {catalog}")
print(f"   📌 Target table : {target_table}")
print(f"   📌 Load type    : FULL (CREATE OR REPLACE)")

In [0]:
%sql
CREATE OR REPLACE TABLE adb_databeli.gold.patient_360 AS

SELECT
    p.patient_id,
    p.first_name,
    p.last_name,
    CAST(p.date_of_birth AS DATE)                                           AS date_of_birth,
    INT(FLOOR(DATEDIFF(CURRENT_DATE(), CAST(p.date_of_birth AS DATE)) / 365.25))  AS age,
    p.gender,
    p.city,
    p.state,
    p.registered_hospital_id                                                AS hospital_id,
    h.hospital_name,
    p.insurance_provider_id,
    ip.provider_name                                                        AS insurance_provider_name,
    COALESCE(lr.total_tests, 0)                                             AS total_tests,
    COALESCE(lr.abnormal_test_count, 0)                                     AS abnormal_test_count,
    COALESCE(lr.critical_test_count, 0)                                     AS critical_test_count,
    lr.last_test_date

FROM adb_databeli.silver.patients  p

LEFT JOIN adb_databeli.silver.hospitals  h
    ON p.registered_hospital_id = h.hospital_id

LEFT JOIN adb_databeli.silver.insurance_providers  ip
    ON p.insurance_provider_id = ip.insurance_provider_id

LEFT JOIN (
    SELECT
        patient_id,
        COUNT(*)                                                AS total_tests,
        COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END) AS abnormal_test_count,
        COUNT(CASE WHEN result_status = 'Critical' THEN 1 END) AS critical_test_count,
        MAX(test_date)                                          AS last_test_date
    FROM adb_databeli.silver.lab_results
    GROUP BY patient_id
) lr
    ON p.patient_id = lr.patient_id

In [0]:
row_count = spark.table(target_table).count()
print(f"✅ Table '{target_table}' built successfully")
print(f"   📊 Total rows: {row_count:,}")

display(spark.table(target_table).limit(10))

In [0]:
dbutils.notebook.exit(str(row_count))